# Lab Day 2 — DeepWeeds · Đoàn Duy Bách (2A202602515)

Notebook chạy lại toàn bộ bài làm. Mọi thí nghiệm đi qua **một** hàm `train.run(Config)` (gọi qua `code/experiments.py`). Kết quả thật của bài nộp được chạy trên **Apple M5 (MPS)**; notebook cũng chạy được trên Colab/Kaggle (CUDA) vì code tự chọn thiết bị `cuda > mps > cpu`.

**Quy tắc:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4** (`step4_final.py` có file khoá).

## 0. Cài đặt môi trường

In [ ]:
# Cài thư viện (Colab/Kaggle). Kaggle: bật Internet trong Settings.
!pip -q install timm openpyxl fvcore

import sys, os, platform
import numpy as np, pandas as pd, torch, timm

# Clone repo bài nộp (Colab/Kaggle). Chạy local thì bỏ qua và đặt REPO_DIR về thư mục repo.
if not os.path.exists("K4-DAY02-DoanDuyBach-2A202602515"):
    !git clone -q https://github.com/DuyBach2003/K4-DAY02-DoanDuyBach-2A202602515.git
REPO_DIR = os.path.abspath("K4-DAY02-DoanDuyBach-2A202602515")
SUB_DIR = f"{REPO_DIR}/submissions/2A202602515_DoanDuyBach"   # thư mục bài nộp (code/, logs/, curves/, ...)
sys.path.insert(0, REPO_DIR)                 # eval.py gốc
sys.path.insert(0, f"{SUB_DIR}/code")        # code đã hoàn thiện

print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else
      ("Apple MPS" if torch.backends.mps.is_available() else "KHÔNG CÓ GPU"))


### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [ ]:
import hashlib

os.chdir(REPO_DIR)
os.makedirs("data/labels", exist_ok=True)
if not os.path.exists("data/images.zip"):
    !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open("data/images.zip", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"

!unzip -q -n data/images.zip -d data/
!ls data | head

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{name}.csv {BASE}/{name}.csv
!ls -la data/labels
# ảnh nằm ngay trong data/ sau khi giải nén -> gom vào data/images
os.makedirs("data/images", exist_ok=True)
!find data -maxdepth 1 -name "*.jpg" -exec mv {} data/images/ \;
!ls data/images | wc -l


In [ ]:
IMAGES_DIR = f"{REPO_DIR}/data/images"
LABELS_DIR = f"{REPO_DIR}/data/labels"
os.chdir(SUB_DIR)   # mọi script chạy từ thư mục bài nộp (đường dẫn ../../data/...)


## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
# EDA + kiểm tra chia dữ liệu (README mục 2.1) -> figures/eda_*.png, logs/step0.json
# + kiểm tra pipeline: loss ban đầu ≈ ln 9, overfit 1 batch, ảnh sau augmentation (figures/sanity_*.png)
!python code/step0_eda_sanity.py --images-dir {IMAGES_DIR} --labels-dir {LABELS_DIR}
import json; s = json.load(open("logs/step0.json"))
print(json.dumps(s["split_check"]["n"]), s["split_check"]["overlap"], "union =", s["split_check"]["union"])
display(pd.DataFrame(s["per_class_table"]))
print("loss ban đầu:", s["initial_loss"], "\noverfit 1 batch:", s["overfit_one_batch"])


In [ ]:
# Unit test tự viết cho các phần dễ sai (focal γ=0 ≡ CE, label smoothing, CutMix λ = diện tích thật, gộp BN,
# precise-BN, temperature scaling, nhóm tham số không weight decay cho norm/bias, EMA, lịch LR)
!cd code && python -m unittest test_code
from IPython.display import Image
display(Image("figures/sanity_augmentations.png")); display(Image("figures/sanity_overfit_one_batch.png"))


## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
# Bước 1: 6 backbone, cùng công thức nền (code/experiments.py: COMMON + BACKBONES), seed 0, 10 epoch.
# Mỗi lần chạy: logs/<exp>/seed0/{config.json, history.csv, summary.json}, curves/<exp>_*.png, predictions/<exp>_seed0_val.csv
!python code/experiments.py B01 B02 B03 B04 B05 B06
# Chẩn đoán: MobileNetV3 huấn luyện FP32 (so với B06 dùng AMP fp16)
!python code/experiments.py D01
# Tốc độ train đo riêng, cùng điều kiện (thời gian/epoch trong log bị nhiễu bởi tải máy)
!python code/bench_train.py
# Chẩn đoán CNN có BN underfit: tính lại thống kê BN bằng ảnh train qua transform val (precise-BN), đo trên VAL
!python code/diag_precise_bn.py


## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
# Bước 2: ablation trên DeiT-S (backbone chọn ở Bước 1). T00 seed 0 = B03 (cùng cấu hình, sao chép).
!python code/experiments.py --alias B03 T00
!python code/experiments.py T01 T02 T03 T04 T05 T06 T07 T08
# kết hợp các yếu tố tốt (xem TRAINING trong code/experiments.py)
!python code/experiments.py T09


## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
# Bước 3: các phương pháp suy luận trên VAL + độ trễ p50/p95/p99 (warmup 10, synchronize, 100 lần đo)
!python code/step3_inference.py --exp T00 --backbone deit_small_patch16_224 --ema-exp T08 --ensemble B03 B04 B02


## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Train cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed; `step4_final.py` chạy test và lưu dự đoán.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
# Bước 4: train thêm seed cho mốc T00 và chung kết F01 (F01 = công thức T09; seed 0 dùng lại lần chạy T09)
!python code/experiments.py T00 --seeds 1 2
!python code/experiments.py --alias T09 F01
!python code/experiments.py F01 --seeds 1 2
# TEST đúng 1 lần mỗi seed (file khoá runs/<exp>/seed<k>/test_done.json); T khớp trên VAL của từng seed
# mốc: công thức nền + 1 view (I00); chung kết: ảnh đầy đủ resize 288, 1 view (I04_288) + temperature scaling
!python code/step4_final.py --exp T00 --backbone deit_small_patch16_224 --seeds 0 1 2 --views none --eval-mode crop --img-size 224
!python code/step4_final.py --exp F01 --backbone deit_small_patch16_224 --seeds 0 1 2 --views none --eval-mode full --img-size 288 --temperature

In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (eval.py gốc, không sửa):
!python {REPO_DIR}/eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out
!python {REPO_DIR}/eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out
!python {REPO_DIR}/eval.py score --pred "predictions/F01uncal_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01uncal --out eval_out


In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất). p95 của cấu hình chung kết (ảnh đầy đủ 288, 1 view, batch 1)
# lấy từ logs/step3_latency.json.
lat = json.load(open("logs/step3_latency.json"))
P95 = lat["I04_res288_b1"]["p95"]
!python {REPO_DIR}/eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" --uncal "predictions/F01uncal_seed*_test.csv" \
    --final-val "predictions/F01_seed*_val.csv" --val-csv {LABELS_DIR}/val_subset0.csv \
    --latency-p95-ms {P95} --latency-method proper \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out eval_out
!mkdir -p logs/eval && cp eval_out/* logs/eval/   # eval_out/ nằm trong .gitignore; logs/eval/ được nộp
# Phân tích sau chung kết, CHỈ trên VAL: tách đóng góp công thức huấn luyện × suy luận (2×2, 3 seed mỗi ô)
!python code/diag_factorial_val.py


## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
# results.xlsx (8 sheet) + biểu đồ phân tích (đánh đổi độ chính xác/độ trễ, ma trận nhầm lẫn, ảnh đoán sai) + curves T00/F01 đủ 3 seed
!python code/make_results.py
!python code/make_figures.py --images-dir {IMAGES_DIR} --labels-dir {LABELS_DIR}
display(pd.read_excel("results.xlsx", sheet_name="Summary", header=None))


## Phần làm thêm (điểm thưởng, RUBRIC mục 2; report mục 10)
Không script nào dùng test fold 0. Nhiều fold dùng đủ bộ ba file của fold đó (S6), test của mỗi fold chạy một lần.

In [ ]:
!pip -q install onnx onnxruntime onnxscript   # chỉ cần cho bonus_onnx.py
!bash code/run_folds.sh                        # F01 trên fold 1, 2 (seed 0) + test 1 lần/fold + eval.py score
!python code/bonus_dinov2_probe.py             # linear probe DINOv2 / DeiT-S đóng băng (train -> val)
!python code/bonus_shift_tta.py shift          # lệch phân phối trên val: macro-F1, ECE trước/sau temperature
!python code/bonus_shift_tta.py bn             # TTA: chuẩn hoá lại thống kê BN (B05, B06)
!python code/bonus_shift_tta.py tent           # TTA: Tent cho F01 (DeiT-S, LayerNorm)
!python code/bonus_attention.py                # Grad-CAM + attention rollout cho lỗi Chinee apple ↔ Snake weed
!python code/bonus_onnx.py                     # xuất ONNX, so độ trễ với PyTorch
!python code/bench_amp_dtype.py                # tốc độ train ResNet-50 theo dtype (căn cứ chọn AMP)
!python code/make_results.py && python code/make_figures.py --images-dir {IMAGES_DIR} --labels-dir {LABELS_DIR}